<a href="https://colab.research.google.com/github/lbayly-dev/CCRB-data/blob/main/NYPD_Officer_360.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import necessary libraries
import requests
import pandas as pd
import io
import matplotlib.pyplot as plt
import seaborn as sns

# Pulling Police Officer csv file from the NYC Open Data Portal
csv_url = 'https://data.cityofnewyork.us/api/views/2fir-qns4/rows.csv?accessType=DOWNLOAD'

try:
    # Send a GET request to the URL
    response = requests.get(csv_url)
    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

    # Read the content into a pandas DataFrame
    df = pd.read_csv(io.StringIO(response.text))

    print(f"Successfully downloaded CSV from {csv_url} and loaded into a DataFrame.")

    # Display the first 5 rows of the DataFrame
    display(df.head())

except requests.exceptions.RequestException as e:
    print(f"Error downloading the CSV file: {e}")
except pd.errors.EmptyDataError:
    print("Error: The downloaded file is empty or not a valid CSV.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")



In [ ]:
# Pulling Allegations csv file from the NYC Open Data Portal
allegations_url = 'https://data.cityofnewyork.us/api/views/6xgr-kwjq/rows.csv?accessType=DOWNLOAD'

try:
    response_allegations = requests.get(allegations_url)
    response_allegations.raise_for_status()
    df_allegations = pd.read_csv(io.StringIO(response_allegations.text))

    print(f"Successfully downloaded CSV from {allegations_url} and loaded into df_allegations.")
    display(df_allegations.head())

except requests.exceptions.RequestException as e:
    print(f"Error downloading the Allegations CSV file: {e}")
except pd.errors.EmptyDataError:
    print("Error: The downloaded Allegations file is empty or not a valid CSV.")
except Exception as e:
    print(f"An unexpected error occurred with Allegations CSV: {e}")

In [ ]:
# Pulling Complaints csv file from the NYC Open Data Portal
complaints_url = 'https://data.cityofnewyork.us/api/views/2mby-ccnw/rows.csv?accessType=DOWNLOAD'

try:
    response_complaints = requests.get(complaints_url)
    response_complaints.raise_for_status()
    df_complaints = pd.read_csv(io.StringIO(response_complaints.text))

    print(f"Successfully downloaded CSV from {complaints_url} and loaded into df_complaints.")
    display(df_complaints.head())

except requests.exceptions.RequestException as e:
    print(f"Error downloading the Complaints CSV file: {e}")
except pd.errors.EmptyDataError:
    print("Error: The downloaded Complaints file is empty or not a valid CSV.")
except Exception as e:
    print(f"An unexpected error occurred with Complaints CSV: {e}")

In [ ]:
# Filter file for active officers
filtered_df = (df['Active Per Last Reported Status'] == 'Yes')

# Convert filtered_df Series to a DataFrame containing only active officers
active_officers_df = df[filtered_df].copy()

# Perform an inner join between active_officers_df and df_allegations on 'Tax ID'
merged_active_officers_allegations = pd.merge(active_officers_df, df_allegations, on='Tax ID', how='inner', suffixes=('_officer', '_allegation'))

print(merged_active_officers_allegations.columns.tolist())

# Perform an inner join between merged_active_officers_allegations and df_complaints on 'Complaint Id'
merged_active_officers_allegations_complaints = pd.merge(merged_active_officers_allegations, df_complaints, on='Complaint Id', how='inner', suffixes=('_allegation_complaint', '_incident_detail'))

print(merged_active_officers_allegations_complaints.columns.tolist())

# Filter file for incidents in a particular borough (Manhattan, Queens, Brooklyn, Bronx, Staten Island)

# Define the desired borough for filtering
desired_borough = 'Manhattan' # You can change this to 'Queens', 'Brooklyn', 'Bronx', 'Staten Island', or any other string to test the validation

valid_boroughs = ['Manhattan', 'Queens', 'Brooklyn', 'Bronx', 'Staten Island']

if desired_borough in valid_boroughs:
    filtered_merged_df = merged_active_officers_allegations_complaints[merged_active_officers_allegations_complaints['Borough Of Incident Occurrence'] == desired_borough].copy()
    print(f"\nFiltering for incidents in {desired_borough}.")
else:
    print(f"\nError: '{desired_borough}' is not a valid borough. Please choose from {', '.join(valid_boroughs)}.")
    filtered_merged_df = pd.DataFrame(columns=merged_active_officers_allegations_complaints.columns) # Create an empty DataFrame with the same columns if input is invalid

print("\nShape of the complaints DataFrame (df_complaints):")
print(df_complaints.shape)
print("Shape of the allegations DataFrame (merged_active_officers_allegations):")
print(merged_active_officers_allegations.shape)
print("Shape of the inner merged DataFrame before penalties:")
print(merged_active_officers_allegations_complaints.shape)
print(f"Shape of the filtered inner merged DataFrame ({desired_borough}):") # Dynamically update this print
print(filtered_merged_df.shape)

In [ ]:
# Print columns in file to decide which columns to keep
print(filtered_merged_df.columns.tolist())

In [ ]:
# Drop the columns that are not needed - can easily be edited to include or remove columns. Code in the line above prints all column names before filtering for reference.
filtered_merged_df = filtered_merged_df.drop(columns=['Victim / Alleged Victim Race (Legacy)', 'Active Per Last Reported Status', 'Last Reported Active Date', 'As Of Date_allegation', 'As Of Date', 'CCRB Investigations Division Recommendation'])

print(filtered_merged_df.columns.tolist())

In [ ]:
# Reformat dates so that they can be read by AirTable, and will be consistently rendered.
date_columns = [
    'As Of Date_officer',
    'Incident Date',
    'CCRB Received Date',
    'Close Date'
]

for col in date_columns:
    if col in filtered_merged_df.columns:
        # Convert to datetime, coercing errors will turn unparseable dates into NaT (Not a Time)
        filtered_merged_df[col] = pd.to_datetime(filtered_merged_df[col], errors='coerce')
        # Format as 'YYYY-MM-DD' string, NaT values will become NaN
        filtered_merged_df[col] = filtered_merged_df[col].dt.strftime('%Y-%m-%d')

print("Date columns formatted to 'YYYY-MM-DD'. Displaying first 5 rows to verify:")
display(filtered_merged_df[date_columns].head())

In [ ]:
# Export the filtered DataFrame to a CSV file. Be sure to rename the file as appropriate if filtering by another borough.
output_manhattan_rows = 'manhattan_police_complaints.csv'
filtered_merged_df.to_csv(output_manhattan_rows, index=False)

# Download the csv file
from google.colab import files
files.download('manhattan_police_complaints.csv')